In [33]:
import os
import random
import textwrap
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from tqdm import tqdm
import google.generativeai as genai
from dotenv import load_dotenv
import json


load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env")

genai.configure(api_key=api_key)

In [34]:
def call_gemini(prompt, model="gemini-1.5-flash"):
    model = genai.GenerativeModel(model)
    resp = model.generate_content(
        contents=prompt
    )
    return resp.text

In [35]:
def generate_descriptions(n=500, model="gemini-2.5-flash"):
    prompt = textwrap.dedent(f"""
        Generate {n} short, realistic financial/business transaction descriptions.
        Each description should be 5-15 words.
        Include purchases, sales, bills, subscriptions, or service transactions.
        Return a JSON array of strings only. For example:
        ["Paid electricity bill online", "Bought a laptop at electronics store"]
    """)
    
    response_text = call_gemini(prompt, model=model)
    
    response_text = response_text.strip().replace("```json", "").replace("```", "")
    descriptions = json.loads(response_text)
    
    if not isinstance(descriptions, list) or len(descriptions) == 0:
        raise ValueError("Gemini returned invalid or empty output.")
    
    return descriptions

descriptions = generate_descriptions(500)
print("Number of AI-generated descriptions:", len(descriptions))
print(descriptions[:5])

Number of AI-generated descriptions: 500
['Bought weekly groceries at local supermarket.', 'Purchased new winter coat from online retailer.', 'Acquired a new smartphone from electronics store.', 'Ordered kitchen appliances for home renovation.', 'Picked up pet supplies at the local store.']


In [39]:
N_RECORDS = 1200
categories = ["Electronics", "Food", "Clothing", "Utilities", "Travel", "Education", "Entertainment"]
payment_methods = ["Card", "Debit", "Bank Transfer", "Wallet"]

rows = []
start_date = datetime(2025, 1, 1)

for i in range(1, N_RECORDS + 1):
    tid = f"TX{i:05d}"
    date = start_date + timedelta(days=random.randint(0, 90))
    customer_id = f"CUST{random.randint(1, 300):04d}"
    amount = round(random.expovariate(1/100), 2)
    category = random.choice(categories)
    payment = random.choice(payment_methods)
    description = random.choice(descriptions)
    
    fraud_risk = int((amount > 300) or (random.random() < 0.05))
    
    rows.append([tid, date.strftime("%Y-%m-%d"), customer_id, amount, category, payment, description, fraud_risk])

df = pd.DataFrame(rows, columns=[
    "transaction_id", "date", "customer_id", "amount", "category", "payment_method", "description", "fraud_risk"
])

os.makedirs('../data', exist_ok=True)
df.to_csv("../data/synthetic_business_dataset_with_fraud.csv", index=False)
print("Dataset shape:", df.shape)
df.head(6)


Dataset shape: (1200, 8)


,transaction_id,date,customer_id,amount,category,payment_method,description,fraud_risk
0,TX00001,2025-01-25,CUST0263,26.07,Electronics,Wallet,Paid for online advertising credits.,0
1,TX00002,2025-01-09,CUST0155,158.41,Electronics,Card,Ordered pet food online.,1
2,TX00003,2025-02-25,CUST0128,180.13,Food,Wallet,Automated marketing platform.,0
3,TX00004,2025-03-25,CUST0079,49.31,Utilities,Wallet,Purchased a yoga class drop-in.,0
4,TX00005,2025-02-04,CUST0224,376.63,Entertainment,Debit,Bought bird seed.,1
5,TX00006,2025-02-08,CUST0025,48.22,Travel,Card,Paid a parking fine.,0
